# Prompt U-Net — GPU Memory Benchmark

Measures peak VRAM on one small and one large 3D volume using `pynvml` (same data as `nvidia-smi`).

**Size definitions** (matching `inference_speed.ipynb`):
- **Small**: ROI in-plane bbox extent ≤128 in both axes (P-UNet single-tile regime, no tiling overhead)
- **Large**: total_voxels > 192³ = 7,077,888 (nnInteractive AutoZoom trigger)

| Section | Content |
|---------|---------|
| **§1** | Scan NPZ test data — find representative small & large volumes |
| **§2** | GPU memory measurement via pynvml |
| **§3** | Results table |

---
## §1 — Find Representative Volumes

In [1]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
project_root = notebook_dir.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
from data.test_data.ds_handler import load_dataset

In [2]:
# Thresholds (matching inference_speed.ipynb)
PATCH_VOXELS = 192 ** 3  # 7,077,888  — Large threshold (nnInteractive AutoZoom)
TILE_LIMIT   = 128       # P-UNet single-tile limit (Small if ROI bbox ≤ this)

NPZ_PATHS = [
    project_root / 'data' / 'test_data' / 'FLARE_2022.npz',
    project_root / 'data' / 'test_data' / 'han_seg_ct.npz',
    project_root / 'data' / 'test_data' / 'han_seg_mri.npz',
    project_root / 'data' / 'test_data' / 'HCCTase_ceCT.npz',
    project_root / 'data' / 'test_data' / 'SegRap2023.npz',
    project_root / 'data' / 'test_data' / 'TotalSeg_mri.npz',
]

# ------------------------------------------------------------------
# Pre-index NPZ files: open once (mmap), build pid→idx, pid→shape, pid→modality
# ------------------------------------------------------------------
_NPZ = {}           # dataset_name → opened npz
_PID_IDX = {}       # (dataset_name, pid) → int index
_PID_SHAPE = {}     # (dataset_name, pid) → (D0, D1, D2)
_PID_MODALITY = {}  # (dataset_name, pid) → str
_SEG_COUNT = {}     # (dataset_name, pid) → int

print('Indexing NPZ files ...')
for npz_path in NPZ_PATHS:
    ds_name = npz_path.stem
    data = np.load(str(npz_path), allow_pickle=False, mmap_mode='r')
    _NPZ[ds_name] = data
    modalities = data['_modalities'] if '_modalities' in data else None
    for i, p in enumerate(data['_pids']):
        p = str(p)
        _PID_IDX[(ds_name, p)] = i
        _PID_SHAPE[(ds_name, p)] = data[f'{i}_image'].shape  # mmap shape, O(1)
        _PID_MODALITY[(ds_name, p)] = str(modalities[i]) if modalities is not None else 'ct'
        _SEG_COUNT[(ds_name, p)] = int(data['_seg_counts'][i])
print(f'  {len(_PID_IDX)} patients indexed')

# ------------------------------------------------------------------
# Cached seg_labels loader (materialises once per pid, reused)
# ------------------------------------------------------------------
_SEG_CACHE = {}

def get_seg_labels(ds_name, pid):
    key = (ds_name, str(pid))
    if key in _SEG_CACHE:
        return _SEG_CACHE[key]
    data = _NPZ.get(ds_name)
    if data is None:
        return None
    i = _PID_IDX.get(key)
    if i is None:
        return None
    seg_count = _SEG_COUNT.get(key, 0)
    if seg_count == 1:
        seg_labels = np.asarray(data[f'{i}_seg_0']).astype(np.int32)
    else:
        seg_labels = np.zeros(_PID_SHAPE[key], dtype=np.int32)
        for j in range(seg_count):
            s = np.asarray(data[f'{i}_seg_{j}'])
            seg_labels[s != 0] = j + 1
    _SEG_CACHE[key] = seg_labels
    return seg_labels

# ------------------------------------------------------------------
# Vectorised ROI in-plane bbox + roi_slices — single bbox computation
# ------------------------------------------------------------------
def roi_bbox_and_slices(seg_labels, axis, roi):
    """Returns (max_bbox_h, max_bbox_w, roi_slices) for one ROI on one axis."""
    mask = np.moveaxis(seg_labels == roi, axis, 0)  # (S, H, W)
    S, H, W = mask.shape

    row_has_fg = np.any(mask, axis=2)  # (S, H)
    col_has_fg = np.any(mask, axis=1)  # (S, W)
    has_fg = np.any(row_has_fg, axis=1)  # (S,)

    first_row = np.argmax(row_has_fg, axis=1)
    last_row  = H - 1 - np.argmax(row_has_fg[:, ::-1], axis=1)
    first_col = np.argmax(col_has_fg, axis=1)
    last_col  = W - 1 - np.argmax(col_has_fg[:, ::-1], axis=1)

    bbox_h = np.where(has_fg, last_row - first_row + 1, 0)
    bbox_w = np.where(has_fg, last_col - first_col + 1, 0)
    roi_slices = int(has_fg.sum())
    return int(bbox_h.max()), int(bbox_w.max()), roi_slices

# ------------------------------------------------------------------
# Scan: for each (patient, axis) use np.bincount to find the largest ROI
# (by total voxel count) in O(n_voxels). Then compute bbox for that one ROI only.
# ~100× faster than iterating all 117+ ROIs per patient for multi-label datasets.
# ------------------------------------------------------------------
candidates = []
print('Scanning volumes (largest ROI per patient per axis) ...')
for npz_path in NPZ_PATHS:
    ds_name = npz_path.stem
    data = _NPZ[ds_name]
    for pid_idx, p in enumerate(data['_pids']):
        p = str(p)
        seg_labels = get_seg_labels(ds_name, p)
        if seg_labels is None:
            continue
        d0, d1, d2 = _PID_SHAPE[(ds_name, p)]
        img_shape = (d0, d1, d2)
        modality = _PID_MODALITY.get((ds_name, p), 'ct')

        # Find the largest ROI (by total voxels) — single pass via bincount
        counts = np.bincount(seg_labels.ravel())
        if len(counts) <= 1:  # only background
            continue
        best_roi = int(np.argmax(counts[1:]) + 1)  # skip background (label 0)

        for axis in range(3):
            h, w = [img_shape[a] for a in range(3) if a != axis]
            max_h, max_w, roi_slices = roi_bbox_and_slices(seg_labels, axis, best_roi)
            if roi_slices == 0:
                continue
            total_voxels = roi_slices * h * w
            max_extent = max(max_h, max_w)
            candidates.append({
                'npz_path': str(npz_path), 'dataset_name': ds_name,
                'pid': p, 'modality': modality,
                'axis': axis, 'roi': best_roi,
                'roi_slices': roi_slices,
                'h': h, 'w': w, 'total_voxels': total_voxels,
                'max_bbox_h': max_h, 'max_bbox_w': max_w,
                'max_bbox_extent': max_extent,
            })

# ---- Classify ----
def classify(c):
    s = c['max_bbox_extent'] <= TILE_LIMIT
    l = c['total_voxels'] > PATCH_VOXELS
    if s and l:   return 'Small & Large'
    if s:         return 'Small'
    if l:         return 'Large'
    return 'Medium'

for c in candidates:
    c['size_bin'] = classify(c)

from collections import Counter
print(f'Scanned {len(candidates)} (patient, axis) combinations (largest ROI only)')
print(f'Size distribution:')
for label, count in Counter(c['size_bin'] for c in candidates).most_common():
    print(f'  {label}: {count}')

Indexing NPZ files ...
  362 patients indexed
Scanning volumes (largest ROI per patient per axis) ...
Scanned 1074 (patient, axis) combinations (largest ROI only)
Size distribution:
  Large: 598
  Small: 276
  Small & Large: 131
  Medium: 69


In [3]:
# Pick: largest Small by total_voxels (stress-test without tiling), largest Large overall
small_candidates = [c for c in candidates if c['size_bin'] == 'Small']
large_candidates = [c for c in candidates if c['size_bin'] == 'Large']

print(f'Small candidates: {len(small_candidates)}  |  Large candidates: {len(large_candidates)}')

rep_small = max(small_candidates, key=lambda c: c['total_voxels'])
rep_large = max(large_candidates, key=lambda c: c['total_voxels'])

for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f"\n{label} volume:")
    print(f"  dataset         : {rep['dataset_name']}")
    print(f"  pid             : {rep['pid']}")
    print(f"  ROI             : {rep['roi']}")
    print(f"  axis            : {rep['axis']}")
    print(f"  roi_slices      : {rep['roi_slices']}")
    print(f"  in-plane        : {rep['h']}×{rep['w']}")
    print(f"  total_voxels    : {rep['total_voxels']:,}")
    print(f"  max_bbox_extent : {rep['max_bbox_extent']}  (≤128 = single tile)")

Small candidates: 276  |  Large candidates: 598

Small volume:
  dataset         : TotalSeg_mri
  pid             : s0085
  ROI             : 47
  axis            : 0
  roi_slices      : 147
  in-plane        : 208×230
  total_voxels    : 7,032,480
  max_bbox_extent : 126  (≤128 = single tile)

Large volume:
  dataset         : TotalSeg_mri
  pid             : s0175
  ROI             : 4
  axis            : 1
  roi_slices      : 129
  in-plane        : 1080×410
  total_voxels    : 57,121,200
  max_bbox_extent : 246  (≤128 = single tile)


---
## §2 — GPU Memory Measurement

In [4]:
import time

import tensorflow as tf

# -------- CRITICAL: prevent TF from pre-allocating all GPU memory --------
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f'{len(gpus)} GPU(s), memory growth enabled')

from inference.inference_volume import VolumeInference
from inference.ssf import ConfidenceDropStrategy

MODEL_PATH = project_root / 'training' / 'p_unet_332.keras'
print(f'Model: {MODEL_PATH}')

2026-05-15 11:37:52.458349: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-15 11:37:52.473016: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778837872.488965     375 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778837872.493627     375 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-15 11:37:52.511316: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

1 GPU(s), memory growth enabled
Model: /home/dpxuser/prompt-unet/training/p_unet_332.keras


In [5]:
def load_volume(rep):
    """Load a 3D volume and isolate the selected ROI."""
    dataset = load_dataset(rep['npz_path'])
    item = dataset[rep['pid']]
    img_3d = np.asarray(item['image']).astype(np.float32)
    segs = item['segmentations']
    if isinstance(segs, list):
        seg_labels = np.zeros_like(img_3d, dtype=np.int32)
        for li, s in enumerate(segs, 1):
            seg_labels[np.asarray(s) != 0] = li
    else:
        seg_labels = np.asarray(segs).astype(np.int32)
    seg_3d_binary = (seg_labels == rep['roi']).astype(np.float32)
    # Pick middle slice containing the ROI as prompt
    sum_axes = tuple(a for a in range(3) if a != rep['axis'])
    areas = seg_3d_binary.sum(axis=sum_axes)
    valid = np.where(areas > 0)[0]
    prompt_idx = valid[len(valid) // 2]
    prompt_2d = np.take(seg_3d_binary, prompt_idx, axis=rep['axis'])
    return img_3d, seg_3d_binary, prompt_2d, prompt_idx


def profile_volume(rep):
    """Run full VolumeInference and return peak VRAM (TF-native tracking).

    Uses tf.config.experimental.get_memory_info('GPU:0')['peak']
    which reports the actual memory used by TF ops, not the driver-level
    allocation pool that nvidia-smi sees.  Requires set_memory_growth(True)
    to prevent TF from pre-allocating all GPU memory.
    """
    img_3d, seg_3d_binary, prompt_2d, prompt_idx = load_volume(rep)
    vi = VolumeInference(
        model_path=str(MODEL_PATH),
        modality=rep['modality'],
        normalization='universal',
        ssf_strategy=ConfidenceDropStrategy(drop_fraction=0.05),
        buffer_size=4,
        batch_size=6,
    )
    # Warm-up — absorbs one-shot autotuning / JIT allocations
    _ = vi.run(img_3d=img_3d, seg_3d_binary=seg_3d_binary,
               initial_prompt_2d_seg=prompt_2d,
               prompt_axis=rep['axis'], prompt_idx=prompt_idx)

    # Reset stats, run, read peak
    tf.config.experimental.reset_memory_stats('GPU:0')
    _ = vi.run(img_3d=img_3d, seg_3d_binary=seg_3d_binary,
               initial_prompt_2d_seg=prompt_2d,
               prompt_axis=rep['axis'], prompt_idx=prompt_idx)
    mem_info = tf.config.experimental.get_memory_info('GPU:0')
    return mem_info['peak'] / (1024 * 1024)  # bytes → MB

In [6]:
results = {}
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    print(f'\nProfiling {label} volume: {rep["pid"]} (roi={rep["roi"]}, axis={rep["axis"]})')
    peak_mb = profile_volume(rep)
    results[label] = peak_mb
    print(f'  Peak VRAM (TF ops): {peak_mb:.1f} MB')
    tf.keras.backend.clear_session()
    tf.config.experimental.reset_memory_stats('GPU:0')


Profiling Small volume: s0085 (roi=47, axis=0)
[VolumeInference] Loading 'p_unet_332.keras' (norm='universal', modality_fallback=MRI, batch_size=6)


I0000 00:00:1778837967.881501     375 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46569 MB memory:  -> device: 0, name: NVIDIA RTX 6000 Ada Generation, pci bus id: 0000:99:00.0, compute capability: 8.9
I0000 00:00:1778837972.068061     425 cuda_dnn.cc:529] Loaded cuDNN version 90500


[VolumeInference] Graph compiled (warm-up done).
  Peak VRAM (TF ops): 248.2 MB

Profiling Large volume: s0175 (roi=4, axis=1)
[VolumeInference] Loading 'p_unet_332.keras' (norm='universal', modality_fallback=MRI, batch_size=6)
[VolumeInference] Graph compiled (warm-up done).
  Peak VRAM (TF ops): 362.4 MB


---
## §3 — Results

In [7]:
P_UNET_PARAMS = 28.0  # million
P_UNET_DISK   = 108   # MB (.keras)

print(f"{'='*66}")
print(f"  Prompt U-Net GPU Memory (TF-native peak)")
print(f"  Small = ROI in-plane bbox ≤ {TILE_LIMIT}   (P-UNet single-tile regime)")
print(f"  Large = total_voxels > {PATCH_VOXELS:,}  (nnInteractive AutoZoom)")
print(f"  Model: {P_UNET_PARAMS:.0f}M params, {P_UNET_DISK} MB on disk")
print(f"  Batch size: 6  |  Memory growth: ON")
print(f"{'='*66}")
print(f"  {'Size':<8} {'Volume':<26} {'bbox_ext':<10} {'total_voxels':<16} {'Peak VRAM':<14}")
print(f"  {'-'*74}")
for label, rep in [('Small', rep_small), ('Large', rep_large)]:
    name = f"{rep['dataset_name']}/{rep['pid']}"
    print(f"  {label:<8} {name:<26} {rep['max_bbox_extent']:<10} {rep['total_voxels']:<16,} {results[label]:.1f} MB")
print(f"{'='*66}")
print(f"\n  Measured via tf.config.experimental.get_memory_info()['peak']")
print(f"  This is TF ops memory, not the nvidia-smi allocation pool.")
print(f"\n  Reference (nnInteractive paper, RTX 4090):")
print(f"    Small  (≤192³) : < 6 GB")
print(f"    Large  (>192³) : < 10 GB")
print(f"  This measurement: RTX A6000 (48 GB)")

  Prompt U-Net GPU Memory (TF-native peak)
  Small = ROI in-plane bbox ≤ 128   (P-UNet single-tile regime)
  Large = total_voxels > 7,077,888  (nnInteractive AutoZoom)
  Model: 28M params, 108 MB on disk
  Batch size: 6  |  Memory growth: ON
  Size     Volume                     bbox_ext   total_voxels     Peak VRAM     
  --------------------------------------------------------------------------
  Small    TotalSeg_mri/s0085         126        7,032,480        248.2 MB
  Large    TotalSeg_mri/s0175         246        57,121,200       362.4 MB

  Measured via tf.config.experimental.get_memory_info()['peak']
  This is TF ops memory, not the nvidia-smi allocation pool.

  Reference (nnInteractive paper, RTX 4090):
    Small  (≤192³) : < 6 GB
    Large  (>192³) : < 10 GB
  This measurement: RTX A6000 (48 GB)
